# **1. 결정 트리**

In [1]:
# 데이터를 로딩하고, 훈련세트와 테스트 세트로 나누기(훈련 세트: 80%)
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(digits.data, digits.target, test_size=0.2, random_state=42)

In [2]:
# 결정 트리 모델을 생성 및 학습하고 정확도 출력하기
model = DecisionTreeClassifier()
model.fit(X_train, y_train)  # 모델 학습
y_pred = model.predict(X_test)  # 예측 수행
accuracy = accuracy_score(y_test, y_pred) # 정확도 계산
print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.8472


In [3]:
# 피처 중요도 출력하기
print("Feature Importances:", model.feature_importances_)  # 피처 중요도 출력

Feature Importances: [0.         0.         0.00554949 0.00223236 0.00738359 0.05604752
 0.00422537 0.         0.         0.00151764 0.03502148 0.
 0.01181288 0.00418895 0.         0.         0.         0.0024747
 0.01693948 0.00970036 0.04828397 0.10083092 0.         0.
 0.00153196 0.00077334 0.07139675 0.07107381 0.00659018 0.00446821
 0.00470776 0.         0.         0.06018675 0.00219114 0.00077334
 0.07801028 0.01672474 0.00226544 0.         0.         0.00470635
 0.1263539  0.04639961 0.00590121 0.00592189 0.01587192 0.
 0.         0.00135335 0.00756542 0.00461551 0.00358885 0.00304182
 0.01724102 0.00230322 0.         0.00887071 0.00558833 0.0044854
 0.06746065 0.03359051 0.         0.0082379 ]


In [4]:
# 테스트 세트의 비율을 30%로 바꾸고 정확도 비교하기
X_train, X_test, y_train, y_test = train_test_split(digits.data, digits.target, test_size=0.3, random_state=42)
model = DecisionTreeClassifier() # 모델 생성
model.fit(X_train, y_train) # 모델 학습
y_pred = model.predict(X_test) # 예측 수행
accuracy = accuracy_score(y_test, y_pred) # 정확도 계산
print(f'70-30 Split Accuracy: {accuracy:.4f}')

70-30 Split Accuracy: 0.8648


In [5]:
# 하이퍼 파라미터(max_depth) 튜닝하기
model = DecisionTreeClassifier(max_depth=3) # 모델 생성
model.fit(X_train, y_train)  # 모델 학습
y_pred = model.predict(X_test)  # 예측 수행
accuracy = accuracy_score(y_test, y_pred) # 정확도 계산
print(f'Max Depth 3 Accuracy: {accuracy:.4f}')

Max Depth 3 Accuracy: 0.4759


In [6]:
# GridSearchCV를 활용한 최적 하이퍼파라미터 찾기
from sklearn.model_selection import GridSearchCV
params = {'max_depth': [3, 5, 10], 'min_samples_split': [2, 4, 6]}
grid_search = GridSearchCV(DecisionTreeClassifier(), params, cv=5, scoring='accuracy') # GridSearchCV 객체 생성
grid_search.fit(X_train, y_train) # GridSearchCV 학습

print("Best Parameters:", grid_search.best_params_)  # 최적 파라미터 출력
print("Best Accuracy:", grid_search.best_score_)  # 최적 정확도 출력

Best Parameters: {'max_depth': 10, 'min_samples_split': 2}
Best Accuracy: 0.8321539239865933


# **2. 앙상블**

## **1) wine 데이터셋 로드 및 훈련/테스트 분리**
wine 데이터를 불러온 후, 훈련 세트와 테스트 세트로 나눠주세요.
*   test_size = 0.25
*   random_state = 11

In [7]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

# 데이터 로드
wine = load_wine()

# 훈련/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(wine.data, wine.target, test_size=0.25, random_state=11)

## **2) SVC, RandomForest, GradientBoosting 모델 학습 및 정확도 평가**
다음 모델을 사용해 개별 학습/예측/정확도 평가를 진행하세요.
*   서포트 벡터 머신: probability=True
*   랜덤 포레스트: n_estimators=100, random_state=11
*   GradientBoostingClassifier: random_state=11




In [8]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

svc_clf = SVC(probability=True) # SVC 모델 정의
rf_clf = RandomForestClassifier(n_estimators=100, random_state=11) # RandomForest 모델 정의
gb_clf = GradientBoostingClassifier(random_state=11) # GradientBoosting 모델 정의

# 모델 학습/예측/정확도 평가
for clf in [svc_clf, rf_clf, gb_clf]:
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    print(f"{clf.__class__.__name__} 정확도: {accuracy_score(y_test, pred):.4f}")

SVC 정확도: 0.8000
RandomForestClassifier 정확도: 0.9778
GradientBoostingClassifier 정확도: 0.9556


## **3) 하드 보팅 분류기 생성 및 평가**
위에서 만든 모델 3개를 기반으로 하드 보팅 방식의 분류기를 만들고, 학습 및 테스트 정확도를 출력하세요.

In [9]:
from sklearn.ensemble import VotingClassifier

# 하드 보팅 분류기 생성
hard_clf = VotingClassifier(estimators=[('svc', svc_clf), ('rf', rf_clf), ('gb', gb_clf)], voting='hard')
hard_clf.fit(X_train, y_train) # 모델 학습
hard_pred = hard_clf.predict(X_test) # 모델 예측
print("Hard Voting Accuracy:", accuracy_score(y_test, hard_pred)) # 정확도 계산

Hard Voting Accuracy: 0.9555555555555556


## **4) 소프트 보팅 분류기 생성 및 평가**
같은 모델 3개로 소프트 보팅 방식의 분류기를 만들어서 학습/평가하세요.

In [10]:
# 소프트 보팅 분류기 생성
soft_clf = VotingClassifier(estimators=[('svc', svc_clf), ('rf', rf_clf), ('gb', gb_clf)], voting='soft')
soft_clf.fit(X_train, y_train) # 모델 학습
soft_pred = soft_clf.predict(X_test) # 모델 예측
print("Soft Voting Accuracy:", accuracy_score(y_test, soft_pred)) # 정확도 계산

Soft Voting Accuracy: 0.9555555555555556


## **5) OOB 평가**
다음 조건에 맞춰 BaggingClassifier를 사용하여 OOB 평가를 수행하세요.

*   Base 모델: GradientBoostingClassifier()
*   n_estimators = 50
*   bootstrap=True

OOB 점수를 출력하세요.

In [11]:
from sklearn.ensemble import BaggingClassifier

# Bagging 모델 생성
bag_clf = BaggingClassifier(GradientBoostingClassifier(), n_estimators=50, bootstrap=True, n_jobs=-1, oob_score=True)
bag_clf.fit(X_train, y_train) # 모델 학습
print("OOB Score:", bag_clf.oob_score_) # OOB 평가 점수 출력

OOB Score: 0.9699248120300752


# **3. 랜덤 포레스트(Random Forest)**

**1) 사이킷런의 load_wine 데이터셋을 RandomForestClassifier을 이용해 예측하시오.**

In [12]:
# 필요한 라이브러리 임포트
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 데이터셋 로드
wine = load_wine()
X = wine.data
y = wine.target

# 데이터셋을 훈련 세트와 테스트 세트로 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 랜덤 포레스트 모델 생성 및 훈련
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 테스트 세트로 예측
y_pred = model.predict(X_test)

# 정확도 평가
accuracy = accuracy_score(y_test, y_pred)
print('랜덤 포레스트 정확도: {0:.4f}'.format(accuracy))

랜덤 포레스트 정확도: 1.0000


**2) load_wine 데이터셋을 이용해 GradientBoostingClassifier 모델을 학습하고, GridSearchCV를 활용해 최적의 하이퍼 파라미터를 찾으시오.**

✅ 조건
1. 데이터셋: load_wine() 사용
2. 훈련 데이터(80%) / 테스트 데이터(20%)로 분리
3. 하이퍼파라미터 튜닝(GridSearchCV 사용)
4. learning_rate: [0.01, 0.1, 0.2]
   n_estimators: [100, 200, 300]
   max_depth: [3, 5, 7]
5. 최적의 하이퍼파라미터를 찾은 후 모델 학습 & 평가
6. 테스트 데이터에서 정확도를 출력

✅ 참고사항

실행 시간이 오래 소요될 수 있습니다!

In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

# 하이퍼파라미터 그리드 설정
params = {
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7]
}

# GradientBoostingClassifier 객체 생성
gb_clf = GradientBoostingClassifier(random_state=42)
# GridSearchCV 객체 생성
grid_cv = GridSearchCV(gb_clf, param_grid=params, cv=5, n_jobs=-1)

# 모델 학습
grid_cv.fit(X_train, y_train)

# 최적 하이퍼파라미터 출력
print('최적 하이퍼파라미터:\n', grid_cv.best_params_)

# 최적 모델로 테스트 데이터 예측
y_pred = grid_cv.best_estimator_.predict(X_test)

# 정확도 출력
accuracy = accuracy_score(y_test, y_pred)
print('예측 정확도: {0:.4f}'.format(accuracy))

최적 하이퍼파라미터:
 {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}
예측 정확도: 0.9444


## **서포트 벡터 머신(SVM)**

**1) 가우시안 RBF 커널을 사용하여 gamma 값이 1이고, 하이퍼파라미터 C의 값이 0.001인 SVM 분류기를 만드시오.**


In [14]:
# 필요한 라이브러리 임포트
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVC
from sklearn.datasets import make_moons

# moons 형태의 데이터 생성
X, y = make_moons(n_samples=200, noise=0.25)

# Pipeline 생성
rbf_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="rbf", gamma=1, C=0.001))
])

# 모델 학습
rbf_kernel_svm_clf.fit(X, y)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svm_clf', SVC(C=0.001, gamma=1))])